# Pipeline Data Mart — Indicateurs de Développement Afrique

Ce notebook exécute l'intégralité du pipeline :

1. **Récupération des pays africains** depuis l'API World Bank
2. **Extraction des indicateurs** (PIB, santé, éducation, etc.) avec cache local
3. **Transformation et nettoyage** des données brutes
4. **Chargement dans DuckDB** en 3 tables (dim_pays, dim_indicateurs, fait_valeurs)
5. **Analyse exploratoire et visualisations** (courbes, carte choroplèthe, scatter)

---
**Source** : World Bank Indicators API v2 (licence CC-BY 4.0)

## Installation des dépendances

Exécutez cette cellule une seule fois si les packages ne sont pas installés :

In [1]:
!pip install requests pandas duckdb pyyaml plotly tenacity jupyter

Defaulting to user installation because normal site-packages is not writeable


---
## Imports et configuration globale

In [2]:
import json
import os
import time
import logging
import warnings
from typing import Optional

import requests
import pandas as pd
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("pipeline")
warnings.filterwarnings("ignore")

print("Tous les imports sont OK.")

Tous les imports sont OK.


### Configuration des chemins et paramètres

Les données brutes sont mises en cache dans `data/raw/`, et la base DuckDB est créée dans `data/processed/`.

In [3]:
PROJECT_DIR = os.path.abspath(os.getcwd())

CACHE_DIR = os.path.join(PROJECT_DIR, "data", "raw")
DB_PATH = os.path.join(PROJECT_DIR, "data", "processed", "africa_datamart.duckdb")
COUNTRIES_CACHE = os.path.join(CACHE_DIR, "countries.json")
CACHE_DURATION = 24 * 3600

# Paramètres modifiables
DATE_START = 2000
DATE_END = 2024
FORCE_REFRESH = False  # Passer à True pour ignorer le cache

print(f"Répertoire projet : {PROJECT_DIR}")
print(f"Cache : {CACHE_DIR}")
print(f"Base DuckDB : {DB_PATH}")

Répertoire projet : c:\Users\fredb\OneDrive - Institut 2IE\PersoProjects\African_Countries_KPI_Analyse\africa-dev-datamart
Cache : c:\Users\fredb\OneDrive - Institut 2IE\PersoProjects\African_Countries_KPI_Analyse\africa-dev-datamart\data\raw
Base DuckDB : c:\Users\fredb\OneDrive - Institut 2IE\PersoProjects\African_Countries_KPI_Analyse\africa-dev-datamart\data\processed\africa_datamart.duckdb


### Configuration des indicateurs

10 indicateurs de développement suivis, répartis en 6 catégories.

In [4]:
INDICATEURS = [
    {"code": "NY.GDP.MKTP.CD",  "name": "PIB (US$ courants)",                 "category": "economie",        "unit": "USD"},
    {"code": "NY.GDP.PCAP.CD",  "name": "PIB par habitant (US$ courants)",    "category": "economie",        "unit": "USD"},
    {"code": "SP.POP.TOTL",     "name": "Population totale",                  "category": "demographie",    "unit": "personnes"},
    {"code": "SH.H2O.BASW.ZS",  "name": "Accès à l'eau potable de base (%)",  "category": "sante",          "unit": "pourcentage"},
    {"code": "EG.ELC.ACCS.ZS",  "name": "Accès à l'électricité (%)",          "category": "infrastructure", "unit": "pourcentage"},
    {"code": "SE.PRM.ENRR",     "name": "Taux brut de scolarisation primaire (%)", "category": "education", "unit": "pourcentage"},
    {"code": "SE.ADT.LITR.ZS",  "name": "Taux d'alphabétisation adultes (%)",  "category": "education",      "unit": "pourcentage"},
    {"code": "SH.DYN.MORT",     "name": "Taux de mortalité infantile (‰)",     "category": "sante",          "unit": "pour_1000"},
    {"code": "SH.XPD.CHEX.GD.ZS", "name": "Dépenses de santé (% du PIB)",      "category": "sante",          "unit": "pourcentage"},
    {"code": "SI.POV.DDAY",     "name": "Taux de pauvreté extrême (%)",         "category": "pauvrete",       "unit": "pourcentage"},
]

print(f"{len(INDICATEURS)} indicateurs configurés")
for i in INDICATEURS:
    print(f"  • {i['code']} — {i['name']} [{i['category']}]")

10 indicateurs configurés
  • NY.GDP.MKTP.CD — PIB (US$ courants) [economie]
  • NY.GDP.PCAP.CD — PIB par habitant (US$ courants) [economie]
  • SP.POP.TOTL — Population totale [demographie]
  • SH.H2O.BASW.ZS — Accès à l'eau potable de base (%) [sante]
  • EG.ELC.ACCS.ZS — Accès à l'électricité (%) [infrastructure]
  • SE.PRM.ENRR — Taux brut de scolarisation primaire (%) [education]
  • SE.ADT.LITR.ZS — Taux d'alphabétisation adultes (%) [education]
  • SH.DYN.MORT — Taux de mortalité infantile (‰) [sante]
  • SH.XPD.CHEX.GD.ZS — Dépenses de santé (% du PIB) [sante]
  • SI.POV.DDAY — Taux de pauvreté extrême (%) [pauvrete]


---
## Partie 1 : Référentiel des pays africains

Récupère la liste des pays africains via l'API World Bank (`/v2/country`), filtre les agrégats régionaux (`region.id == "NA"`), et met en cache le résultat.

In [5]:
def _load_countries_cache() -> Optional[list[dict]]:
    if os.path.exists(COUNTRIES_CACHE):
        with open(COUNTRIES_CACHE, encoding="utf-8") as f:
            return json.load(f)
    return None

def _save_countries_cache(data: list[dict]) -> None:
    os.makedirs(CACHE_DIR, exist_ok=True)
    with open(COUNTRIES_CACHE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def _fetch_countries_from_api() -> list[dict]:
    url = "https://api.worldbank.org/v2/region/AFR/country?format=json&per_page=300"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    if not isinstance(data, list) or len(data) < 2:
        raise ValueError("Format de réponse API inattendu")
    return data[1]

def _is_african_country(entry: dict) -> bool:
    region = entry.get("region")
    if not region or not isinstance(region, dict):
        return False
    if region.get("id") == "NA":
        return False
    return True

def get_african_countries(use_cache: bool = True) -> list[dict]:
    if use_cache:
        cached = _load_countries_cache()
        if cached:
            logger.info(f"Chargement de {len(cached)} pays depuis le cache")
            return cached

    raw = _fetch_countries_from_api()
    african = [c for c in raw if _is_african_country(c)]
    result = []
    for c in african:
        result.append({
            "code_iso3": c.get("id"),
            "code_iso2": c.get("iso2Code"),
            "nom": c.get("name"),
            "region": c.get("region", {}).get("value"),
            "sous_region": c.get("adminregion", {}).get("value") if c.get("adminregion") else None,
            "niveau_revenu": c.get("incomeLevel", {}).get("value") if c.get("incomeLevel") else None,
            "capitale": c.get("capitalCity"),
            "latitude": c.get("latitude"),
            "longitude": c.get("longitude"),
        })
    _save_countries_cache(result)
    return result

In [6]:
pays = get_african_countries(use_cache=not FORCE_REFRESH)
country_codes = [p["code_iso3"] for p in pays if p["code_iso3"]]

print(f"Pays africains trouvés : {len(country_codes)}")
print(f"Exemples : {[p['nom'] for p in pays[:10]]}")

Pays africains trouvés : 54
Exemples : ['Angola', 'Burundi', 'Benin', 'Burkina Faso', 'Botswana', 'Central African Republic', "Cote d'Ivoire", 'Cameroon', 'Congo, Dem. Rep.', 'Congo, Rep.']


---
## Partie 2 : Extraction des données API

Pour chaque indicateur configuré, appelle l'API World Bank (`/v2/country/{codes}/indicator/{code}`) avec pagination complète et retry exponentiel (3 tentatives). Les réponses sont mises en cache 24h.

In [7]:
def _cache_path(indicator_code: str) -> str:
    return os.path.join(CACHE_DIR, f"{indicator_code}.json")

def _is_cache_valid(indicator_code: str) -> bool:
    path = _cache_path(indicator_code)
    if not os.path.exists(path):
        return False
    return (time.time() - os.path.getmtime(path)) < CACHE_DURATION

def _read_cache(indicator_code: str) -> Optional[dict]:
    if _is_cache_valid(indicator_code):
        with open(_cache_path(indicator_code), encoding="utf-8") as f:
            return json.load(f)
    return None

def _write_cache(indicator_code: str, data: dict) -> None:
    os.makedirs(CACHE_DIR, exist_ok=True)
    with open(_cache_path(indicator_code), "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=30),
    retry=retry_if_exception_type((requests.RequestException, ConnectionError)),
)
def _call_api(url: str) -> dict:
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    return resp.json()

In [8]:
def _fetch_indicator_from_api(
    country_codes: list[str], indicator_code: str,
    date_start: int, date_end: int,
) -> dict:
    codes_joined = ";".join(country_codes)
    base_url = f"https://api.worldbank.org/v2/country/{codes_joined}/indicator/{indicator_code}"
    params = {"date": f"{date_start}:{date_end}", "format": "json", "per_page": 20000}
    page = 1
    all_data = []

    while True:
        params["page"] = page
        url = base_url + "?" + "&".join(f"{k}={v}" for k, v in params.items())
        response = _call_api(url)
        if not isinstance(response, list) or len(response) < 2:
            break
        metadata, page_data = response[0], response[1]
        if page_data is None:
            break
        all_data.extend(page_data)
        if page >= metadata.get("pages", 1):
            break
        page += 1

    return [{"total": len(all_data), "page": 1, "pages": 1, "per_page": len(all_data)}, all_data]

def fetch_indicator(
    country_codes: list[str], indicator_code: str,
    date_start: int = 2000, date_end: int = 2024,
    force_refresh: bool = False,
) -> dict:
    if not force_refresh:
        cached = _read_cache(indicator_code)
        if cached is not None:
            logger.info(f"Cache utilisé pour {indicator_code}")
            return cached
    logger.info(f"Extraction de {indicator_code} depuis l'API...")
    data = _fetch_indicator_from_api(country_codes, indicator_code, date_start, date_end)
    _write_cache(indicator_code, data)
    n = len(data[1]) if isinstance(data[1], list) else 0
    logger.info(f"Récupéré {n} valeurs pour {indicator_code}")
    return data

### Exécution de l'extraction

La cellule suivante extrait les 10 indicateurs pour tous les pays africains. Cela peut prendre 2–5 minutes selon le réseau.

In [9]:
raw_dataframes = []
for ind in INDICATEURS:
    code = ind["code"]
    raw = fetch_indicator(
        country_codes=country_codes,
        indicator_code=code,
        date_start=DATE_START,
        date_end=DATE_END,
        force_refresh=FORCE_REFRESH,
    )
    raw_dataframes.append(raw)

print("\nExtraction terminée pour tous les indicateurs.")

00:50:17 [INFO] Extraction de NY.GDP.MKTP.CD depuis l'API...
00:50:32 [INFO] Récupéré 1350 valeurs pour NY.GDP.MKTP.CD
00:50:32 [INFO] Extraction de NY.GDP.PCAP.CD depuis l'API...
00:50:54 [INFO] Récupéré 1350 valeurs pour NY.GDP.PCAP.CD
00:50:54 [INFO] Extraction de SP.POP.TOTL depuis l'API...
00:51:03 [INFO] Récupéré 1350 valeurs pour SP.POP.TOTL
00:51:03 [INFO] Extraction de SH.H2O.BASW.ZS depuis l'API...
00:51:18 [INFO] Récupéré 1350 valeurs pour SH.H2O.BASW.ZS
00:51:18 [INFO] Extraction de EG.ELC.ACCS.ZS depuis l'API...
00:51:28 [INFO] Récupéré 1350 valeurs pour EG.ELC.ACCS.ZS
00:51:28 [INFO] Extraction de SE.PRM.ENRR depuis l'API...
00:51:31 [INFO] Récupéré 1350 valeurs pour SE.PRM.ENRR
00:51:31 [INFO] Extraction de SE.ADT.LITR.ZS depuis l'API...
00:51:34 [INFO] Récupéré 1350 valeurs pour SE.ADT.LITR.ZS
00:51:34 [INFO] Extraction de SH.DYN.MORT depuis l'API...
00:51:36 [INFO] Récupéré 1350 valeurs pour SH.DYN.MORT
00:51:36 [INFO] Extraction de SH.XPD.CHEX.GD.ZS depuis l'API...
00


Extraction terminée pour tous les indicateurs.


---
## Partie 3 : Transformation et nettoyage

Convertit chaque réponse JSON brute en DataFrame pandas avec les colonnes normalisées (`pays_code`, `pays_nom`, `annee`, `valeur`, `indicateur_code`, `is_missing`). Détecte et signale les doublons.

In [10]:
def clean_indicator_data(raw_json: dict) -> pd.DataFrame:
    colonnes = ["pays_code", "pays_nom", "annee", "valeur", "indicateur_code", "is_missing"]

    if not isinstance(raw_json, list) or len(raw_json) < 2 or not raw_json[1]:
        return pd.DataFrame(columns=colonnes)

    df = pd.json_normalize(raw_json[1])
    mapping = {
        "countryiso3code": "pays_code",
        "country.value": "pays_nom",
        "date": "annee",
        "value": "valeur",
        "indicator.id": "indicateur_code",
    }
    df = df[[c for c in mapping if c in df.columns]].rename(columns=mapping)
    df = df[df["pays_code"].notna()].copy()
    df["annee"] = pd.to_numeric(df["annee"], errors="coerce").astype("Int64")
    df["valeur"] = pd.to_numeric(df["valeur"], errors="coerce").astype(float)
    df["is_missing"] = df["valeur"].isna()

    doublons = df.duplicated(subset=["pays_code", "indicateur_code", "annee"], keep=False)
    if doublons.any():
        logger.warning(f"{doublons.sum()} doublons détectés et dédupliqués")
        df = df.drop_duplicates(subset=["pays_code", "indicateur_code", "annee"], keep="first")

    return df

def merge_all_indicators(dataframes: list[pd.DataFrame]) -> pd.DataFrame:
    non_empty = [df for df in dataframes if not df.empty]
    if not non_empty:
        return pd.DataFrame(columns=["pays_code", "pays_nom", "annee", "valeur", "indicateur_code", "is_missing"])
    return pd.concat(non_empty, ignore_index=True)

In [11]:
cleaned = [clean_indicator_data(r) for r in raw_dataframes]
df_valeurs = merge_all_indicators(cleaned)

total = len(df_valeurs)
manquantes = df_valeurs["is_missing"].sum()
print(f"Lignes totales : {total}")
print(f"Valeurs manquantes : {manquantes} ({manquantes/total*100:.1f}%)")
print(f"Indicateurs distincts : {df_valeurs['indicateur_code'].nunique()}")
print(f"Pays distincts : {df_valeurs['pays_code'].nunique()}")

Lignes totales : 13500
Valeurs manquantes : 2860 (21.2%)
Indicateurs distincts : 10
Pays distincts : 54


In [12]:
pct_par_indicateur = df_valeurs.groupby("indicateur_code")["is_missing"].mean().mul(100).round(1)
print("% valeurs manquantes par indicateur :")
pct_par_indicateur

% valeurs manquantes par indicateur :


indicateur_code
EG.ELC.ACCS.ZS        5.2
NY.GDP.MKTP.CD        2.2
NY.GDP.PCAP.CD        2.2
SE.ADT.LITR.ZS       81.7
SE.PRM.ENRR          25.5
SH.DYN.MORT           0.0
SH.H2O.BASW.ZS        2.7
SH.XPD.CHEX.GD.ZS     7.0
SI.POV.DDAY          85.3
SP.POP.TOTL           0.0
Name: is_missing, dtype: float64

---
## Partie 4 : Construction du Data Mart DuckDB

Charge les données dans une base DuckDB avec 3 tables :
- `dim_pays` : code_iso3, nom, region, sous_region, niveau_revenu, latitude, longitude
- `dim_indicateurs` : code, name, category, unit
- `fait_valeurs` : pays_code, indicateur_code, annee, valeur, is_missing

L'opération est **idempotente** (`CREATE OR REPLACE TABLE`).

In [13]:
def build_dim_pays(pays: list[dict]) -> pd.DataFrame:
    records = []
    for p in pays:
        records.append({
            "code_iso3": p["code_iso3"],
            "nom": p["nom"],
            "region": p["region"],
            "sous_region": p["sous_region"],
            "niveau_revenu": p["niveau_revenu"],
            "latitude": p["latitude"],
            "longitude": p["longitude"],
        })
    return pd.DataFrame(records)

def build_dim_indicateurs(indicateurs: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(indicateurs)

def load_to_duckdb(
    df_pays: pd.DataFrame,
    df_indicateurs: pd.DataFrame,
    df_valeurs: pd.DataFrame,
    db_path: str,
) -> None:
    os.makedirs(os.path.dirname(db_path), exist_ok=True)
    conn = duckdb.connect(db_path)
    try:
        conn.execute("CREATE OR REPLACE TABLE dim_pays AS SELECT * FROM df_pays")
        conn.execute("CREATE OR REPLACE TABLE dim_indicateurs AS SELECT * FROM df_indicateurs")
        conn.execute("CREATE OR REPLACE TABLE fait_valeurs AS SELECT * FROM df_valeurs")
        n_pays, n_inds, n_vals = conn.execute(
            "SELECT (SELECT COUNT(*) FROM dim_pays), (SELECT COUNT(*) FROM dim_indicateurs), (SELECT COUNT(*) FROM fait_valeurs)"
        ).fetchone()
        logger.info(f"Chargement terminé : {n_pays} pays, {n_inds} indicateurs, {n_vals} valeurs")
    finally:
        conn.close()

In [14]:
df_pays = build_dim_pays(pays)
df_indicateurs = build_dim_indicateurs(INDICATEURS)

load_to_duckdb(df_pays, df_indicateurs, df_valeurs, DB_PATH)
print(f"Base DuckDB créée : {DB_PATH}")

00:51:42 [INFO] Chargement terminé : 54 pays, 10 indicateurs, 13500 valeurs


Base DuckDB créée : c:\Users\fredb\OneDrive - Institut 2IE\PersoProjects\African_Countries_KPI_Analyse\africa-dev-datamart\data\processed\africa_datamart.duckdb


---
## Partie 5 : Analyse exploratoire et visualisations

Les visualisations utilisent Plotly Express et sont interactives.

In [15]:
conn = duckdb.connect(DB_PATH)
df_pays_db = conn.execute("SELECT * FROM dim_pays").fetchdf()
df_inds_db = conn.execute("SELECT * FROM dim_indicateurs").fetchdf()
df_vals_db = conn.execute("SELECT * FROM fait_valeurs").fetchdf()

print(f"Pays : {len(df_pays_db)}")
print(f"Indicateurs : {len(df_inds_db)}")
print(f"Valeurs : {len(df_vals_db)}")
print(f"Période : {int(df_vals_db['annee'].min())} – {int(df_vals_db['annee'].max())}")

Pays : 54
Indicateurs : 10
Valeurs : 13500
Période : 2000 – 2024


### 5.1 Taux de complétude par indicateur

In [16]:
completude = (
    df_vals_db.groupby("indicateur_code")
    .agg(total=("valeur", "count"), manquantes=("is_missing", "sum"))
    .reset_index()
)
completude["taux_completude"] = (1 - completude["manquantes"] / (completude["total"] + completude["manquantes"])) * 100
completude = completude.merge(df_inds_db, left_on="indicateur_code", right_on="code")
completude[["name", "category", "total", "manquantes", "taux_completude"]].sort_values("taux_completude", ascending=False)

,name,category,total,manquantes,taux_completude
9,Population totale,demographie,1350,0,100.000000
5,Taux de mortalité infantile (‰),sante,1350,0,100.000000
2,PIB par habitant (US$ courants),economie,1320,30,97.777778
1,PIB (US$ courants),economie,1320,30,97.777778
6,Accès à l'eau potable de base (%),sante,1313,37,97.259259
0,Accès à l'électricité (%),infrastructure,1280,70,94.814815
7,Dépenses de santé (% du PIB),sante,1256,94,93.037037
4,Taux brut de scolarisation primaire (%),education,1006,344,74.518519
3,Taux d'alphabétisation adultes (%),education,247,1103,18.296296
8,Taux de pauvreté extrême (%),pauvrete,198,1152,14.666667


### 5.2 Évolution du PIB par habitant — Afrique de l'Ouest

In [17]:
pays_ouest = ["CIV", "SEN", "GHA", "NGA", "BFA", "MLI", "BEN", "TGO"]
query = f"""
SELECT f.pays_code, f.annee, f.valeur, p.nom
FROM fait_valeurs f
JOIN dim_pays p ON f.pays_code = p.code_iso3
WHERE f.indicateur_code = 'NY.GDP.PCAP.CD'
  AND f.pays_code IN {tuple(pays_ouest)}
ORDER BY f.annee
"""
df_pib = conn.execute(query).fetchdf()

fig = px.line(
    df_pib, x="annee", y="valeur", color="nom",
    title="Évolution du PIB par habitant — Afrique de l'Ouest",
    labels={"annee": "Année", "valeur": "PIB par habitant (US$ courants)", "nom": "Pays"},
    markers=True,
)
fig.update_layout(template="plotly_white")
fig.show()

### 5.3 Carte choroplèthe — Accès à l'électricité

Dernière année disponible pour l'indicateur.

In [18]:
query = """
SELECT f.pays_code, f.annee, f.valeur, p.nom
FROM fait_valeurs f
JOIN dim_pays p ON f.pays_code = p.code_iso3
WHERE f.indicateur_code = 'EG.ELC.ACCS.ZS'
  AND f.annee = (SELECT MAX(annee) FROM fait_valeurs WHERE indicateur_code = 'EG.ELC.ACCS.ZS')
"""
df_elec = conn.execute(query).fetchdf()

if df_elec.empty:
    print("Aucune donnée disponible pour cet indicateur.")
else:
    fig = px.choropleth(
        df_elec,
        locations="pays_code",
        locationmode="ISO-3",
        color="valeur",
        hover_name="nom",
        hover_data={"valeur": True, "annee": True, "pays_code": False},
        color_continuous_scale=px.colors.sequential.Blues,
        projection="mercator",
        title=f"Accès à l'électricité — {int(df_elec['annee'].iloc[0])}",
        labels={"valeur": "Accès (%)"},
    )
    fig.update_layout(
        template="plotly_white",
        geo=dict(
            showframe=False,
            showcoastlines=True,
            coastlinecolor="gray",
            showland=True,
            landcolor="lightgray",
            showocean=True,
            oceancolor="#f0f8ff",
            lataxis=dict(range=[-35, 40]),
            lonaxis=dict(range=[-20, 55]),
        ),
    )
    fig.show()

### 5.4 Corrélation : Accès à l'eau potable vs Mortalité infantile

In [19]:
query = """
WITH eau AS (
    SELECT pays_code, annee, valeur AS acces_eau
    FROM fait_valeurs WHERE indicateur_code = 'SH.H2O.BASW.ZS'
),
mortalite AS (
    SELECT pays_code, annee, valeur AS mortalite
    FROM fait_valeurs WHERE indicateur_code = 'SH.DYN.MORT'
)
SELECT eau.pays_code, eau.annee, eau.acces_eau, mortalite.mortalite, p.nom
FROM eau
JOIN mortalite ON eau.pays_code = mortalite.pays_code AND eau.annee = mortalite.annee
JOIN dim_pays p ON eau.pays_code = p.code_iso3
WHERE eau.acces_eau IS NOT NULL AND mortalite.mortalite IS NOT NULL AND eau.annee = 2020
"""
df_corr = conn.execute(query).fetchdf()

fig = px.scatter(
    df_corr,
    x="acces_eau", y="mortalite", text="nom",
    title="Corrélation : Accès à l'eau potable vs Mortalité infantile (2020)",
    labels={"acces_eau": "Accès à l'eau potable (%)", "mortalite": "Mortalité infantile (‰)"},
    trendline="ols",
)
fig.update_traces(textposition="top center")
fig.update_layout(template="plotly_white")
fig.show()

### 5.5 Pays avec le plus de valeurs manquantes

Identifier les lacunes dans les données est essentiel pour un décideur.

In [20]:
manquantes_par_pays = (
    df_vals_db[df_vals_db["is_missing"]]
    .groupby("pays_code").size()
    .sort_values(ascending=False)
    .reset_index(name="nb_manquantes")
)
manquantes_par_pays = manquantes_par_pays.merge(
    df_pays_db[["code_iso3", "nom"]], left_on="pays_code", right_on="code_iso3"
)
print("Pays avec le plus de valeurs manquantes :")
manquantes_par_pays[["nom", "nb_manquantes"]].head(10)

Pays avec le plus de valeurs manquantes :


,nom,nb_manquantes
0,South Sudan,140
1,Eritrea,90
2,"Somalia, Fed. Rep.",86
3,Libya,79
4,Djibouti,73
5,Equatorial Guinea,71
6,Gabon,69
7,Liberia,67
8,Zimbabwe,66
9,Guinea-Bissau,66


---
## Export JSON pour le dashboard Next.js

Génère les fichiers JSON statiques dans `frontend/public/data/`.

In [21]:
EXPORT_DIR = os.path.join(PROJECT_DIR, "frontend", "public", "data")
os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(os.path.join(EXPORT_DIR, "valeurs_par_indicateur"), exist_ok=True)

# Pays
pays_export = conn.execute("SELECT * FROM dim_pays ORDER BY nom").fetchdf().to_dict(orient="records")
with open(os.path.join(EXPORT_DIR, "pays.json"), "w", encoding="utf-8") as f:
    json.dump(pays_export, f, indent=2, ensure_ascii=False)

# Indicateurs
inds_export = conn.execute("SELECT * FROM dim_indicateurs ORDER BY code").fetchdf().to_dict(orient="records")
with open(os.path.join(EXPORT_DIR, "indicateurs.json"), "w", encoding="utf-8") as f:
    json.dump(inds_export, f, indent=2, ensure_ascii=False)

# Valeurs (complet)
vals_export = conn.execute("SELECT * FROM fait_valeurs ORDER BY pays_code, indicateur_code, annee").fetchdf().to_dict(orient="records")
with open(os.path.join(EXPORT_DIR, "valeurs.json"), "w", encoding="utf-8") as f:
    json.dump(vals_export, f, indent=2, ensure_ascii=False)

# Valeurs par indicateur
codes = conn.execute("SELECT DISTINCT indicateur_code FROM fait_valeurs").fetchall()
for (code,) in codes:
    subset = conn.execute(
        f"SELECT * FROM fait_valeurs WHERE indicateur_code = '{code}' ORDER BY pays_code, annee"
    ).fetchdf().to_dict(orient="records")
    with open(os.path.join(EXPORT_DIR, "valeurs_par_indicateur", f"{code}.json"), "w", encoding="utf-8") as f:
        json.dump(subset, f, indent=2, ensure_ascii=False)

print(f"Export terminé : {EXPORT_DIR}")
print(f"  • {len(pays_export)} pays")
print(f"  • {len(inds_export)} indicateurs")
print(f"  • {len(vals_export)} valeurs")
print(f"  • {len(codes)} fichiers par indicateur")

Export terminé : c:\Users\fredb\OneDrive - Institut 2IE\PersoProjects\African_Countries_KPI_Analyse\africa-dev-datamart\frontend\public\data
  • 54 pays
  • 10 indicateurs
  • 13500 valeurs
  • 10 fichiers par indicateur


---
## Fermeture de la connexion

In [22]:
conn.close()
print("Connexion DuckDB fermée.")
print("Pipeline terminé avec succès.")

Connexion DuckDB fermée.
Pipeline terminé avec succès.
